In [148]:
'''import sys
import platform

print("Python version:")
print(sys.version)

print("\nPython executable path:")
print(sys.executable)

print("\nPlatform info:")
print(platform.platform())'''

'import sys\nimport platform\n\nprint("Python version:")\nprint(sys.version)\n\nprint("\nPython executable path:")\nprint(sys.executable)\n\nprint("\nPlatform info:")\nprint(platform.platform())'

In [149]:
''' import os
from pathlib import Path
import re
import pandas as pd
import urllib.request
import tarfile

# -------------------------------
# 1. Download and extract IMDb dataset
# -------------------------------
url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
filename = "aclImdb_v1.tar.gz"

if not os.path.exists("aclImdb"):
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, filename)
    print("Extracting dataset...")
    with tarfile.open(filename, "r:gz") as tar:
        tar.extractall()
print("Dataset ready!")

# -------------------------------
# 2. Function to load dataset into DataFrame
# -------------------------------
def load_imdb_to_df(path):
    texts = []
    labels = []
    for label in ["pos", "neg"]:
        folder = Path(path)/label
        for file in folder.iterdir():
            text = file.read_text(encoding='utf-8').lower()
            text = re.sub(r"<.*?>", "", text)  # remove HTML tags
            texts.append(text)
            labels.append(1 if label=="pos" else 0)
    df = pd.DataFrame({"review": texts, "label": labels})
    return df

# -------------------------------
# 3. Create train and test CSV
# -------------------------------
train_df = load_imdb_to_df("aclImdb/train")
test_df = load_imdb_to_df("aclImdb/test")

train_df.to_csv("imdb_train.csv", index=False)
test_df.to_csv("imdb_test.csv", index=False)

print("CSV files saved: imdb_train.csv and imdb_test.csv")
'''

' import os\nfrom pathlib import Path\nimport re\nimport pandas as pd\nimport urllib.request\nimport tarfile\n\n# -------------------------------\n# 1. Download and extract IMDb dataset\n# -------------------------------\nurl = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"\nfilename = "aclImdb_v1.tar.gz"\n\nif not os.path.exists("aclImdb"):\n    print("Downloading dataset...")\n    urllib.request.urlretrieve(url, filename)\n    print("Extracting dataset...")\n    with tarfile.open(filename, "r:gz") as tar:\n        tar.extractall()\nprint("Dataset ready!")\n\n# -------------------------------\n# 2. Function to load dataset into DataFrame\n# -------------------------------\ndef load_imdb_to_df(path):\n    texts = []\n    labels = []\n    for label in ["pos", "neg"]:\n        folder = Path(path)/label\n        for file in folder.iterdir():\n            text = file.read_text(encoding=\'utf-8\').lower()\n            text = re.sub(r"<.*?>", "", text)  # remove HTML tags\

In [150]:
import torch as tp 
import torch.nn as nn 
import torch.optim as optim
from torch.utils.data import DataLoader,Dataset
import pandas as pd 
import numpy as np 
from tqdm import tqdm

In [151]:
#--------------------
# TODO: Load Dataset
#--------------------
train_df=pd.read_csv('imdb_train.csv')
test_df=pd.read_csv('imdb_test.csv')


subset_size=15000
train_df=train_df.sample(subset_size,random_state=42).reset_index(drop=True)


In [152]:
 # TODO: Convert words to number
from collections import Counter

words=[]
for text in train_df['review']:
    for word in text.lower().split():
        words.append(word)

words_count=Counter(words)

vocab={}
index=1

for word,_ in words_count.most_common(10000):
    vocab[word]=index
    index+=1

print("Vocabulary size:", len(vocab))
print(list(vocab.items())[:10])


Vocabulary size: 10000
[('the', 1), ('a', 2), ('and', 3), ('of', 4), ('to', 5), ('is', 6), ('in', 7), ('i', 8), ('this', 9), ('that', 10)]


In [153]:
train_df['label']

0        1
1        0
2        1
3        0
4        0
        ..
14995    0
14996    1
14997    1
14998    0
14999    0
Name: label, Length: 15000, dtype: int64

In [154]:
 # TODO: Convert Reviews into fixed length
MAX_LEN=200
def text_to_number(text):
    number=[]
    for word in text.lower().split():
        number.append(vocab.get(word,0))

    if len(number)<MAX_LEN:
        zeros=[0]*(MAX_LEN-len(number))
        number=zeros+number

    else:
        number=number[:MAX_LEN]

    return number

In [155]:
len(text_to_number('brilliant over-acting by lesley ann warren. '))


200

In [156]:
 # TODO: Prepare to train

X=[]
y=[]

for _,row in train_df.iterrows():
    X.append(text_to_number(row['review']))
    y.append(row['label'])
X=tp.tensor(X,dtype=tp.long)
y=tp.tensor(y,dtype=tp.float32)



In [157]:
print(X.shape)
print(y.shape)

torch.Size([15000, 200])
torch.Size([15000])


In [158]:
class IMDBDataset(Dataset):
    def __init__(self,X,y):
        self.X=X
        self.y=y

    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, index):
        return self.X[index],self.y[index]

In [159]:
dataset=IMDBDataset(X,y)

dataloader=DataLoader(
    dataset,
    batch_size=8,
    shuffle=True
)

In [160]:
 # TODO: Training a RNN MODEL
class SimpleRNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding=nn.Embedding(10001,64)
        self.rnn=nn.LSTM(64,64,batch_first=True,dropout=0.2)
        self.fc=nn.Linear(64,1)
        self.sigmoid=nn.Sigmoid()

    def forward(self,X):
        X=self.embedding(X)
        out,(h_n,c_n)=self.rnn(X)
        out=out[:,-1,:]
        out=self.fc(out)
        return self.sigmoid(out).squeeze()

In [161]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

# Step 1: Split train / validation
X_train_t, X_val_t, y_train_t, y_val_t = train_test_split(
    X, y, test_size=0.2, random_state=42
)

train_dataset = IMDBDataset(X_train_t, y_train_t)
val_dataset = IMDBDataset(X_val_t, y_val_t)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

# Step 2: Define model, loss, optimizer
model = SimpleRNN()
loss_fn = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Step 3: Early stopping setup
best_val_loss = float('inf')
patience = 3
counter = 0
best_model_state = None

epochs = 20

# Step 4: Training loop with validation
for epoch in range(epochs):
    # ---- TRAIN ----
    model.train()
    running_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        pred = model(inputs)
        loss = loss_fn(pred, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)

    avg_train_loss = running_loss / len(train_loader.dataset)

    # ---- VALIDATION ----
    model.eval()
    val_loss_total = 0
    with tp.no_grad():
        for inputs, labels in val_loader:
            pred = model(inputs)
            loss = loss_fn(pred, labels)
            val_loss_total += loss.item() * inputs.size(0)

    avg_val_loss = val_loss_total / len(val_loader.dataset)

    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # ---- EARLY STOPPING ----
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        counter = 0
        best_model_state = model.state_dict()  # keep best in memory
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered!")
            break

# Load best model from memory
model.load_state_dict(best_model_state)
print("Training complete. Best model loaded in memory!")

tp.save(model.state_dict(), "best_simple_rnn.pth")
print("Model saved as best_simple_rnn.pth")


c:\Users\Admin\anaconda3\envs\ml\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


Epoch 1 | Train Loss: 0.6891 | Val Loss: 0.6885
Epoch 2 | Train Loss: 0.6116 | Val Loss: 0.5512
Epoch 3 | Train Loss: 0.4383 | Val Loss: 0.4716
Epoch 4 | Train Loss: 0.3167 | Val Loss: 0.4225
Epoch 5 | Train Loss: 0.2222 | Val Loss: 0.4588
Epoch 6 | Train Loss: 0.1457 | Val Loss: 0.5225
Epoch 7 | Train Loss: 0.0830 | Val Loss: 0.5955
Early stopping triggered!
Training complete. Best model loaded in memory!
Model saved as best_simple_rnn.pth


In [162]:
'''model=SimpleRNN()
loss_fn=nn.BCELoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)
epochs=20

for epoch in range(epochs):
    running_loss=0
    for inputs,labels in dataloader:
        optimizer.zero_grad()

        pred=model(inputs)
        loss=loss_fn(pred,labels)

        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)  # multiply by batch size

    avg_loss = running_loss / len(dataloader.dataset)

    print(f'Epoch: {epoch+1}, Loss: {avg_loss} ')
    '''

"model=SimpleRNN()\nloss_fn=nn.BCELoss()\noptimizer=optim.Adam(model.parameters(),lr=0.001)\nepochs=20\n\nfor epoch in range(epochs):\n    running_loss=0\n    for inputs,labels in dataloader:\n        optimizer.zero_grad()\n\n        pred=model(inputs)\n        loss=loss_fn(pred,labels)\n\n        loss.backward()\n        optimizer.step()\n        running_loss += loss.item() * inputs.size(0)  # multiply by batch size\n\n    avg_loss = running_loss / len(dataloader.dataset)\n\n    print(f'Epoch: {epoch+1}, Loss: {avg_loss} ')\n    "

In [163]:
# 1️⃣ Recreate the model structure
model = SimpleRNN()

# 2️⃣ Load the saved weights
model.load_state_dict(tp.load("best_simple_rnn.pth"))

# 3️⃣ Switch to evaluation mode
model.eval()


C:\Users\Admin\AppData\Local\Temp\ipykernel_11804\3829806430.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(tp.load("best_simple_rnn.pth"))


SimpleRNN(
  (embedding): Embedding(10001, 64)
  (rnn): LSTM(64, 64, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [164]:
# Convert test reviews to sequences
X_val_seq = [text_to_number(text) for text in test_df["review"]]
y_val_seq = list(test_df["label"])

# Convert to tensors
X_val_tensor = tp.tensor(X_val_seq, dtype=tp.long)
y_val_tensor = tp.tensor(y_val_seq, dtype=tp.float32)

# Create Dataset and DataLoader
val_dataset = IMDBDataset(X_val_tensor, y_val_tensor)
val_loader = DataLoader(val_dataset, batch_size=8)


In [165]:
correct = 0
total = 0

with tp.no_grad():  # no gradients needed
    for inputs, labels in val_loader:
        outputs = model(inputs)
        predicted = (outputs >= 0.5).float()  # threshold 0.5
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total
print(f"Test Accuracy: {accuracy*100:.2f}%")


Test Accuracy: 80.00%


In [166]:
def predict_sentiment(text):
    model.eval()
    seq = text_to_number(text)  # convert text to numbers
    tensor = tp.tensor([seq], dtype=tp.long)
    with tp.no_grad():
        output = model(tensor)
        return "Positive" if output.item() >= 0.5 else "Negative"

# Example usage
print(predict_sentiment("This movie was amazing and I loved it"))
print(predict_sentiment("I hated this movie. It was boring"))


Positive
Negative
